# Day 4 v2 — Model 10: PhoBERT-base-v2 Improved (top-8, R-Drop, EMA 0.9999)

**Architecture:** `vinai/phobert-base-v2` (RoBERTa VN, 12L, 768d, 135M)
— Underthesea word segmentation → partial unfreeze top **8**/12 layers → mean_pooling → price head + aux category head.

**Improvements vs NB07 (session 23 — research-backed):**
| Technique | NB07 (old) | NB10 (improved) |
|---|---|---|
| keep_top_layers | 4/12 (33%) | **8/12 (67%)** — optimal for 269K samples |
| EMA decay | 0.999 (window ~1K steps) | **0.9999** (window ~10K steps) |
| warmup_ratio | 0.1 (full epoch ramp-up) | **0.05** (half epoch) |
| weight_decay | 0.02 | **0.01** (BERT standard) |
| llrd_decay | 0.9 | **0.85** (wider LR range for 8 layers) |
| batch_size | 64 | **48** (reduced for R-Drop 2x forward) |
| R-Drop | — | **alpha=0.3** (MSE consistency loss) |

**Note:** Underthesea word segmentation cache (~2-3h lần đầu) được share với NB07/08.
Cache path: `cache/phobert_seg_{train,val,test}.pkl`

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
uv add underthesea
```

Restart kernel sau khi sync xong.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.phobert_model import PhoBERTRunner, PHOBERT_BASE

MODEL_NAME = PHOBERT_BASE  # "vinai/phobert-base-v2"
CACHE_DIR   = Path("cache")
WEIGHT_DIR  = Path("weights")
VAL_PRED_DIR = Path("val_predictions")

# Improved hyperparams (session 23)
KEEP_TOP      = 8       # top-8/12 layers = 67% encoder
BATCH         = 48      # reduced from 64 for R-Drop 2x forward
BASE_LR       = 2e-5
WEIGHT_DECAY  = 0.01    # was 0.02 — BERT standard
LLRD_DECAY    = 0.85    # was 0.9 — wider range for 8 layers
EPOCHS        = 10
PATIENCE      = 3
EMA_DECAY     = 0.9999  # was 0.999 — window ~10K steps
WARMUP_RATIO  = 0.05    # was 0.1 — half epoch warmup
R_DROP_ALPHA  = 0.3     # R-Drop MSE consistency loss weight

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).


torch: 2.9.0+cu128 | cuda: True
GPU: NVIDIA GeForce RTX 3090 Ti


## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 269,112 | Val: 3,926 | Test: 3,872


## 2. Underthesea Word Segmentation Cache

PhoBERT tokenizer yêu cầu text đã qua word segmentation (Underthesea).
Cache shared với NB07/NB08 — **nếu NB07 đã chạy trước, cache đã có sẵn** (~pkl files).
Nếu chưa có: tự động chạy word_segment (~2-3h trên CPU cho 269K samples).

In [3]:
CACHE_DIR.mkdir(exist_ok=True)
SEG_TRAIN = CACHE_DIR / "phobert_seg_train.pkl"
SEG_VAL   = CACHE_DIR / "phobert_seg_val.pkl"
SEG_TEST  = CACHE_DIR / "phobert_seg_test.pkl"

print(f"Cache train: {'EXISTS' if SEG_TRAIN.exists() else 'MISSING — will compute ~2h'}")
print(f"Cache val:   {'EXISTS' if SEG_VAL.exists() else 'MISSING — will compute ~5 min'}")
print(f"Cache test:  {'EXISTS' if SEG_TEST.exists() else 'MISSING — will compute ~5 min'}")

Cache train: MISSING — will compute ~2h
Cache val:   MISSING — will compute ~5 min
Cache test:  MISSING — will compute ~5 min


## 3. Setup — top-8/12 layers (67% encoder unfrozen)

- Underthesea word-segment → tokenize all data
- LLRD: head lr=2e-5, mỗi layer thấp hơn × decay=0.85
- Trainable params: ~70M encoder + ~1M heads ≈ 71M total
- Expected VRAM: ~10-14GB (batch=48, top-8, R-Drop 2x forward)

In [4]:
runner = PhoBERTRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=KEEP_TOP,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
    train_seg_cache=SEG_TRAIN,
    val_seg_cache=SEG_VAL,
)

Categories (8): [np.str_('Bách Hóa'), np.str_('Làm Đẹp - Sức Khỏe'), np.str_('Mẹ và Bé'), np.str_('Nhà Cửa - Đời Sống'), np.str_('Thời Trang'), np.str_('Ô Tô - Xe Máy'), np.str_('Điện Lạnh và Gia Dụng'), np.str_('Điện Tử - Công Nghệ')]
Word-segmenting 269,112 texts (Underthesea)...


word_segment: 100%|██████████| 269112/269112 [10:18<00:00, 435.13it/s]


Cached to cache/phobert_seg_train.pkl
Word-segmenting 3,926 texts (Underthesea)...


word_segment: 100%|██████████| 3926/3926 [00:08<00:00, 457.97it/s]


Cached to cache/phobert_seg_val.pkl
Loading tokenizer: vinai/phobert-base-v2


...

Tokenizing train (269,112) — ~2-3 min ...
Tokenizing val (3,926) ...
Target: mean=5.4417, std=0.8904 | norm range [-4.10, 1.65]
Loading model: vinai/phobert-base-v2


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Params: 56,907,785/135,203,081 trainable | top 8/12 layers unfrozen | hidden=768
Device: cuda


## 4. Train

10 epochs, early stopping patience=3.
Val MAE evaluated on full 3926 samples per epoch using EMA model.
R-Drop: 2x forward per step → consistency regularization.

Expected: ~20-30 min/epoch on RTX 3090 Ti (batch=48, top-8 layers).

In [5]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
)

Train: batch=48 | steps/ep=5607 | total=56070 | warmup=2803
LLRD: base=2.0e-05, decay=0.85, emb_lr=2.84e-06 | EMA=0.9999 | Huber delta=1.0 | R-Drop=0.3


Epoch 1/10:   0%|          | 6/5607 [00:01<23:31,  3.97it/s, loss=0.6292]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch  1/10 (1343s) | train_loss=0.2813 | val_loss=0.2881 | val_mae=153.05k | lr=1.99e-05
  ** best val_mae=153.05k


Epoch  2/10 (1343s) | train_loss=0.1504 | val_loss=0.1987 | val_mae=127.27k | lr=1.88e-05
  ** best val_mae=127.27k


...

Epoch  6/10 (1338s) | train_loss=0.0478 | val_loss=0.1108 | val_mae=84.46k | lr=7.55e-06
  ** best val_mae=84.46k


Epoch  7/10 (1339s) | train_loss=0.0404 | val_loss=0.1097 | val_mae=83.43k | lr=4.53e-06
  ** best val_mae=83.43k


Epoch  8/10 (1338s) | train_loss=0.0361 | val_loss=0.1095 | val_mae=82.95k | lr=2.11e-06
  ** best val_mae=82.95k


Epoch  9/10 (1336s) | train_loss=0.0335 | val_loss=0.1094 | val_mae=82.69k | lr=5.42e-07
  ** best val_mae=82.69k


Epoch 10/10 (1339s) | train_loss=0.0325 | val_loss=0.1095 | val_mae=82.55k | lr=0.00e+00
  ** best val_mae=82.55k
Restored best EMA: val_mae=82.55k


## 5. Training History

In [6]:
plot_training_history(history, title="PhoBERT-base Improved (top-8, R-Drop, EMA 0.9999)")

## 6. Save Weights + Val Predictions + Test Predictions

In [7]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "phobert_base_improved.pth"))
print("Saved weights/phobert_base_improved.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "phobert_base_improved_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/phobert_base_improved_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test, seg_cache=SEG_TEST)
with open(VAL_PRED_DIR / "phobert_base_improved_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/phobert_base_improved_test.json ({len(test_preds)} samples)")

Saved weights/phobert_base_improved.pth
Running val predictions (3926 samples)...


Saved val_predictions/phobert_base_improved_val.json (3926 samples)
Running test predictions (3872 samples)...
Word-segmenting 3,872 texts (Underthesea)...


word_segment: 100%|██████████| 3872/3872 [00:07<00:00, 507.52it/s]


Cached to cache/phobert_seg_test.pkl


Saved val_predictions/phobert_base_improved_test.json (3872 samples)


## 7. Evaluate on 200 Test Samples

In [8]:
def phobert_improved_pricer(item):
    return runner.inference(item)

results = evaluate(phobert_improved_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

  0%|          | 0/200 [00:00<?, ?it/s]

115 28 11 70 46 5 175 42 48 34 19 20 283 42 179 62 14 36 100 24 2 26 5 35 44 10 98 23 627 29 38 16 6 37 70 11 25 11 341 200 74 50 133 4 47 297 19 4 23 30 23 121 66 212 129 76 109 14 16 322 105 80 14 54 52 53 32 100 110 3 39 27 18 76 2 19 12 16 69 32 47 566 43 166 46 156 11 7 59 26 59 8 62 415 36 29 37 179 28 4 113 60 60 165 106 44 15 14 16 16 178 61 204 333 52 10 112 341 1 12 98 41 118 51 46 14 23 6 291 4 43 14 141 4 113 311 316 35 199 67 62 58 45 0 247 1 121 65 12 20 355 24 13 7 18 139 59 32 8 3 18 97 84 13 22 140 37 57 38 31 128 18 8 88 10 40 129 4 37 93 44 3 14 51 42 18 29 550 395 96 31 79 55 89 1 7 15 59 23 21 


MAE: 77.6k VND | MSE: 16,767 | R2: 70.8%


## 8. Sanity Check

In [9]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "phobert_base_improved.pth"), map_location="cpu", weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"keep_top_layers={ckpt['keep_top_layers']} | model_name={ckpt['model_name']}")

Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L 
Actual:  479.0k VND
Predict: 363.7k VND
Error:   115.3k VND
Checkpoint keys: ['ema_state_dict', 'y_mean', 'y_std', 'model_name', 'keep_top_layers', 'cat_classes']
keep_top_layers=8 | model_name=vinai/phobert-base-v2
